# SIS #2 — Pastebin Protocol Implementation (Python)

**Step 3: Working implementation** · gRPC/Protocol Buffers vs REST/JSON

Based on SIS #1: `POST /v1/snippets`, `GET /v1/snippets/{id}`, `GET /s/{id}`; immutable snippets, TTL 60 s–30 days, maximum 1 MiB. Shared SQLite persistence and business logic.




## 1. Install dependencies
Run once, then restart the kernel if Jupyter asks.

In [ ]:
%pip install -q "grpcio>=1.60" "grpcio-tools>=1.60" "protobuf>=4.25" "fastapi>=0.110" "uvicorn>=0.27" "httpx>=0.27" "pydantic>=2"

## 2. Configuration and Protocol Buffers contract
The `.proto` file is generated on disk, then compiled to Python stubs. The gRPC request/response fields match the REST API.

In [ ]:
from pathlib import Path
import sys, subprocess, uuid, time, json, sqlite3, secrets, string, threading, socket
from datetime import datetime, timezone

WORKDIR = Path.cwd() / "sis2_pastebin_runtime"
WORKDIR.mkdir(exist_ok=True)
DB_PATH = WORKDIR / "pastebin.sqlite3"
GRPC_PORT, REST_PORT = 50071, 18080
MAX_BYTES = 1_048_576
MIN_TTL, MAX_TTL, DEFAULT_TTL = 60, 30 * 24 * 3600, 7 * 24 * 3600
BASE_URL = f"http://127.0.0.1:{REST_PORT}"

PROTO = r'''syntax = "proto3";
package pastebin;
service Pastebin {
  rpc CreateSnippet(CreateSnippetRequest) returns (CreateSnippetResponse);
  rpc GetSnippet(GetSnippetRequest) returns (GetSnippetResponse);
}
message CreateSnippetRequest {
  string content = 1;
  string title = 2;
  string language = 3;
  int32 ttl_seconds = 4;
  string idempotency_key = 5;
}
message CreateSnippetResponse {
  string id = 1;
  string url = 2;
  string created_at = 3;
  string expires_at = 4;
}
message GetSnippetRequest { string id = 1; }
message GetSnippetResponse {
  string id = 1;
  string content = 2;
  string title = 3;
  string language = 4;
  string created_at = 5;
  string expires_at = 6;
}
'''
(WORKDIR / "pastebin.proto").write_text(PROTO, encoding="utf-8")
subprocess.run([sys.executable, "-m", "grpc_tools.protoc", f"-I{WORKDIR}", f"--python_out={WORKDIR}", f"--grpc_python_out={WORKDIR}", str(WORKDIR / "pastebin.proto")], check=True)
sys.path.insert(0, str(WORKDIR))
import pastebin_pb2 as pb
import pastebin_pb2_grpc as pb_grpc
print("Protocol Buffers stubs generated in", WORKDIR)

### Разбор блока 2: конфигурация и протокол

**Задача блока:** подготовить общие настройки и описать контракт gRPC — какие операции доступны, что клиент передаёт и что получает в ответ. Здесь серверы ещё не запускаются.

**1. Импорты и рабочие файлы.** `Path` нужен для путей; `sys` — для выбора Python и настройки импорта; `subprocess` — для запуска компилятора протокола. Остальные модули понадобятся дальше: `uuid` для ключей идемпотентности, `time` для времени, `json` для сериализации, `sqlite3` для базы, `secrets` и `string` для случайных ID, `threading` для потоков. `datetime` и `timezone` помогают представить время в UTC. `socket` импортирован, но далее не используется.

`Path.cwd()` возвращает текущую рабочую папку ядра, которая не обязательно совпадает с папкой самого ноутбука. В ней создаётся `sis2_pastebin_runtime`; `exist_ok=True` разрешает использовать уже существующую папку. `DB_PATH` задаёт путь к базе SQLite; саму базу откроет следующий блок.

**2. Настройки.** `GRPC_PORT = 50071` и `REST_PORT = 18080` разделяют два сервера по портам. `MAX_BYTES = 1_048_576` — ровно 1 МиБ; подчёркивания в числе нужны только для читаемости. TTL (*time to live*) — срок жизни записи: минимум 60 секунд, максимум 30 дней, по умолчанию 7 дней. Выражения с `24 * 3600` переводят дни в секунды. `BASE_URL` — адрес REST-сервера на текущем компьютере; `f"...{REST_PORT}"` подставляет порт в строку.

**3. Строка `PROTO`.** Это текст на языке Protocol Buffers, а не Python-код. Тройные кавычки позволяют записать несколько строк; префикс `r` отключает обычную обработку обратных слешей в Python-строке. `syntax = "proto3"` задаёт версию синтаксиса, `package pastebin` — пространство имён контракта.

`service Pastebin` объявляет две операции удалённого вызова (RPC): `CreateSnippet` создаёт запись, `GetSnippet` читает её. В скобках указано сообщение запроса, после `returns` — сообщение ответа. `message` описывает структуру данных, `string` означает строку, `int32` — знаковое 32-битное целое. Числа после `=` — номера полей для двоичного кодирования, **не значения по умолчанию**.

| Сообщение | Что содержит |
|---|---|
| `CreateSnippetRequest` | Текст `content`, заголовок `title`, язык `language`, TTL `ttl_seconds` и ключ повтора `idempotency_key`. |
| `CreateSnippetResponse` | ID, публичный URL и время создания/истечения срока. Сам текст здесь не возвращается. |
| `GetSnippetRequest` | Только ID нужной записи. |
| `GetSnippetResponse` | ID, текст, заголовок, язык и обе временные метки. |

У обычных скалярных полей `proto3` при отсутствии значения строка читается как `''`, число — как `0`. В блоке gRPC эти значения частично заменяются настройками по умолчанию. Сам контракт не проверяет предел 1 МиБ или допустимый TTL: это делает общий сервис.

**4. Генерация Python-модулей.** `write_text(..., encoding="utf-8")` сохраняет контракт в `pastebin.proto`. `subprocess.run(...)` запускает `grpc_tools.protoc` через тот же Python, в котором работает ядро (`sys.executable`). Ключ `-I` задаёт каталог поиска `.proto`, `--python_out` — каталог классов сообщений, `--grpc_python_out` — каталог кода gRPC. `check=True` прерывает выполнение с исключением, если компилятор завершится с ошибкой.

Получаются два модуля: `pastebin_pb2` содержит сообщения и их сериализацию, `pastebin_pb2_grpc` — клиентский интерфейс и основу серверного обработчика. `sys.path.insert(0, ...)` позволяет импортировать их из рабочей папки; `as pb` и `as pb_grpc` задают короткие имена. Для изменения контракта нужно править `PROTO`, затем заново генерировать модули.


## 3. SQLite storage and shared business logic


In [ ]:
import hashlib

class ServiceError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status, self.message = status, message

class PastebinService:
    def __init__(self, db_path):
        self.db_path = str(db_path)
        self._lock = threading.RLock()
        with self._connect() as db:
            db.execute("PRAGMA journal_mode=WAL")
            db.execute("""CREATE TABLE IF NOT EXISTS snippets (
                id TEXT PRIMARY KEY, content TEXT NOT NULL, title TEXT,
                language TEXT NOT NULL, size_bytes INTEGER NOT NULL,
                created_at REAL NOT NULL, expires_at REAL NOT NULL)""")
            db.execute("CREATE INDEX IF NOT EXISTS snippets_expires_idx ON snippets(expires_at)")
            db.execute("""CREATE TABLE IF NOT EXISTS idempotency_keys (
                key TEXT PRIMARY KEY, fingerprint TEXT NOT NULL,
                response_json TEXT NOT NULL, created_at REAL NOT NULL)""")

    def _connect(self):
        db = sqlite3.connect(self.db_path, timeout=5, check_same_thread=False)
        db.row_factory = sqlite3.Row
        return db

    @staticmethod
    def _iso(ts):
        return datetime.fromtimestamp(ts, timezone.utc).isoformat().replace('+00:00', 'Z')

    def create(self, content, title='', language='plaintext', ttl_seconds=DEFAULT_TTL, key=None):
        if not isinstance(content, str):
            raise ServiceError(422, 'content must be a string')
        size = len(content.encode('utf-8'))
        if size > MAX_BYTES:
            raise ServiceError(413, 'content exceeds 1 MiB')
        if size < 1:
            raise ServiceError(422, 'content must not be empty')
        if not isinstance(ttl_seconds, int) or not MIN_TTL <= ttl_seconds <= MAX_TTL:
            raise ServiceError(422, 'ttl_seconds out of range')
        if not isinstance(title, str) or len(title) > 200:
            raise ServiceError(422, 'invalid title')
        if not isinstance(language, str) or not 1 <= len(language) <= 32:
            raise ServiceError(422, 'invalid language')
        if key is not None:
            try: key = str(uuid.UUID(key))
            except (ValueError, TypeError, AttributeError):
                raise ServiceError(422, 'idempotency key must be a UUID')
        payload = dict(content=content, title=title, language=language, ttl_seconds=ttl_seconds)
        fingerprint = hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        with self._lock, self._connect() as db:
            db.execute('BEGIN IMMEDIATE')
            if key:
                old = db.execute('SELECT fingerprint,response_json FROM idempotency_keys WHERE key=?', (key,)).fetchone()
                if old:
                    if old['fingerprint'] != fingerprint:
                        raise ServiceError(409, 'idempotency key reused with different content')
                    return json.loads(old['response_json'])
            now = time.time()
            for _ in range(3):
                snippet_id = ''.join(secrets.choice(string.ascii_letters + string.digits) for _ in range(8))
                try:
                    db.execute('INSERT INTO snippets VALUES (?,?,?,?,?,?,?)',
                               (snippet_id, content, title, language, size, now, now+ttl_seconds))
                    break
                except sqlite3.IntegrityError:
                    continue
            else:
                raise ServiceError(503, 'unable to allocate unique ID')
            response = dict(id=snippet_id, url=f'{BASE_URL}/s/{snippet_id}',
                            created_at=self._iso(now), expires_at=self._iso(now+ttl_seconds))
            if key:
                db.execute('INSERT INTO idempotency_keys VALUES (?,?,?,?)',
                           (key, fingerprint, json.dumps(response), now))
            db.commit()
            return response

    def get(self, snippet_id):
        with self._connect() as db:
            row = db.execute('SELECT * FROM snippets WHERE id=? AND expires_at>?',
                             (snippet_id, time.time())).fetchone()
        if row is None:
            raise ServiceError(404, 'snippet not found or expired')
        return dict(id=row['id'], content=row['content'], title=row['title'],
                    language=row['language'], created_at=self._iso(row['created_at']),
                    expires_at=self._iso(row['expires_at']))

    def cleanup(self):
        with self._lock, self._connect() as db:
            result = db.execute('DELETE FROM snippets WHERE id IN (SELECT id FROM snippets WHERE expires_at <= ? LIMIT 1000)', (time.time(),))
            db.commit()
            return result.rowcount

service = PastebinService(DB_PATH)
print('Storage ready:', DB_PATH)

### Разбор блока 3: хранение и общая бизнес-логика

**Задача блока:** создать один сервис, который хранит записи и применяет правила независимо от способа обращения. Позже и REST, и gRPC будут вызывать один объект `service`. Сниппет здесь — сохранённый фрагмент текста; операции изменения его содержимого нет.

**1. `ServiceError`.** Собственное исключение хранит числовой статус и понятное сообщение. `super().__init__(message)` передаёт сообщение базовому классу `Exception`. Сервис сообщает об ошибке, а REST- и gRPC-обработчики позже переводят её в свой формат ответа.

**2. Инициализация `PastebinService`.** `__init__` сохраняет путь и создаёт `threading.RLock()` — блокировку для согласования операций записи из разных потоков этого объекта. `PRAGMA journal_mode=WAL` включает журнал SQLite, который облегчает совместную работу чтения и записи, но не делает количество параллельных писателей неограниченным.

Таблица `snippets` хранит уникальный `id`, текст, заголовок, язык, размер в байтах и две временные метки. `PRIMARY KEY` запрещает повтор ID, `NOT NULL` запрещает SQL-значение `NULL`. Индекс по `expires_at` помогает искать истёкшие записи. Таблица `idempotency_keys` хранит уникальный ключ запроса, отпечаток его параметров, ранее сформированный JSON-ответ и время создания. `IF NOT EXISTS` позволяет повторно открыть уже подготовленную базу.

**3. Вспомогательные методы.** `_connect()` открывает SQLite-соединение с ожиданием блокировки до 5 секунд. `check_same_thread=False` снимает проверку использования соединения только создавшим его потоком; само по себе это не заменяет синхронизацию. `sqlite3.Row` позволяет обращаться к столбцам по имени, например `row['content']`.

Конструкция `with ... as db` управляет транзакцией соединения: при нормальном выходе фиксирует её, при исключении откатывает. У SQLite этот контекстный менеджер **не закрывает само соединение**. `_iso()` переводит Unix-время в строку UTC с окончанием `Z`. Декоратор `@staticmethod` означает, что этому методу не нужен `self`. Начальный `_` в именах методов — соглашение о внутреннем использовании.

**4. Проверка данных в `create()`.** Значения после `=` в сигнатуре — аргументы по умолчанию. Дальше проверки идут последовательно:

- `content` должен быть строкой. `encode('utf-8')` превращает её в байты, а `len(...)` считает размер: один символ может занимать несколько байтов. Размер больше 1 МиБ даёт 413, пустая строка — 422. Строка из пробелов здесь считается непустой.
- TTL должен пройти проверку типа `int` и находиться в диапазоне от 60 секунд до 30 дней включительно, иначе возвращается 422.
- Заголовок должен быть строкой длиной до 200 символов, язык — строкой длиной от 1 до 32 символов. Эти длины считаются в символах, а не байтах.
- Если ключ передан, `uuid.UUID(key)` проверяет его формат, а `str(...)` приводит к единому представлению. Неподходящий ключ даёт 422.

**5. Идемпотентность.** Идемпотентный повтор позволяет безопасно отправить тот же запрос ещё раз и получить тот же результат без второй записи. Словарь `payload` содержит текст, заголовок, язык и TTL. `json.dumps(..., sort_keys=True, ensure_ascii=False)` сериализует его с устойчивым порядком ключей, `hashlib.sha256(...).hexdigest()` вычисляет отпечаток. Это не шифрование текста; отпечаток нужен для сравнения параметров.

После захвата блокировки выполняется `BEGIN IMMEDIATE`: SQLite начинает транзакцию с намерением записи. Если ключ уже сохранён, одинаковый отпечаток возвращает прежний ответ через `json.loads(...)`, а другой отпечаток даёт 409. Например, повтор с текстом `first` вернёт прежний ID, но замена текста на `different` с тем же ключом вызовет конфликт. Сравниваются все поля `payload`, включая TTL, а не только содержимое.

**6. Создание записи.** `time.time()` даёт текущее Unix-время в секундах. Генератор собирает ID из 8 случайных латинских букв и цифр через `secrets.choice`. Запрос `INSERT` передаёт значения отдельно через `?`: это параметризованный SQL, а не вставка пользовательского текста в команду.

Если возникает `sqlite3.IntegrityError`, цикл пробует ещё раз; всего предусмотрено три попытки. `else` у `for` выполняется, если цикл завершился без `break`: тогда выдаётся 503. При успехе формируется словарь с ID, публичным URL и датами. Если есть ключ, его отпечаток и ответ сохраняются в той же транзакции; `commit()` фиксирует запись и ключ вместе.

**7. Чтение и очистка.** `get()` ищет ID только с `expires_at > текущее время`. Поэтому запись с истёкшим TTL уже недоступна, даже если физически остаётся в базе. Отсутствующая или истёкшая запись даёт 404; найденная превращается в словарь с текстом и метаданными.

`cleanup()` удаляет до 1000 истёкших сниппетов за вызов и возвращает число удалённых строк. Этот метод только объявлен: автоматического расписания очистки в ноутбуке нет. Таблицу ключей он не очищает, поэтому повтор с прежним ключом может вернуть старый ответ даже после истечения TTL сниппета.

Последняя строка `service = PastebinService(DB_PATH)` создаёт общий экземпляр и подготавливает базу. Здесь находится основная логика; сетевые слои лишь принимают запросы и вызывают её.


## 4. REST/JSON server (FastAPI)
Same business operations as the gRPC service. REST is the baseline; HTTP errors map to appropriate status codes.

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import PlainTextResponse
from pydantic import BaseModel, Field
import uvicorn

app = FastAPI(title='SIS2 Pastebin REST baseline')

class CreateBody(BaseModel):
    content: str
    title: str = ''
    language: str = 'plaintext'
    ttl_seconds: int = DEFAULT_TTL

def rest_call(fn, *args, **kwargs):
    try: return fn(*args, **kwargs)
    except ServiceError as e: raise HTTPException(status_code=e.status, detail=e.message)

@app.post('/v1/snippets', status_code=201)
def create_rest(body: CreateBody, idempotency_key: str | None = Header(default=None)):
    return rest_call(service.create, body.content, body.title, body.language, body.ttl_seconds, idempotency_key)

@app.get('/v1/snippets/{snippet_id}')
def get_rest(snippet_id: str):
    return rest_call(service.get, snippet_id)

@app.get('/s/{snippet_id}', response_class=PlainTextResponse)
def get_public(snippet_id: str):
    return rest_call(service.get, snippet_id)['content']

rest_server = None
rest_thread = None

def start_rest():
    global rest_server, rest_thread
    if rest_server is not None and rest_server.started:
        return
    config = uvicorn.Config(app, host='127.0.0.1', port=REST_PORT, log_level='warning', access_log=False)
    rest_server = uvicorn.Server(config)
    rest_thread = threading.Thread(target=rest_server.run, daemon=True)
    rest_thread.start()
    for _ in range(100):
        if rest_server.started: return
        time.sleep(.05)
    raise RuntimeError('REST server did not start')

start_rest()
print('REST listening at', BASE_URL, '| docs:', BASE_URL+'/docs')

### Разбор блока 4: REST-сервер на FastAPI

**Задача блока:** предоставить общий сервис через HTTP. Для запросов и обычных ответов используется JSON, а публичный маршрут возвращает простой текст.

**1. Приложение и тело запроса.** `FastAPI(...)` создаёт приложение, `title` задаёт его название в документации. `CreateBody(BaseModel)` — модель Pydantic: поле `content` обязательно, остальные получают указанные значения по умолчанию. FastAPI разбирает JSON и запускает проверку модели до вызова обработчика; затем правила размера и TTL проверяет `service.create`. Pydantic также может приводить некоторые входные значения к объявленным типам. Импорт `Field` в этом блоке не используется.

**2. Перевод ошибок.** `rest_call(fn, *args, **kwargs)` принимает функцию и её аргументы. `*args` собирает позиционные аргументы, `**kwargs` — именованные; в `fn(*args, **kwargs)` они передаются дальше. Если сервис выбрасывает `ServiceError`, обёртка создаёт `HTTPException` с тем же статусом и сообщением в поле `detail`. Здесь перехватываются именно ошибки сервиса, а не любые исключения Python.

**3. Маршруты.** Декоратор `@app...` связывает HTTP-метод и URL с функцией под ним.

| Запрос | Результат |
|---|---|
| `POST /v1/snippets` | Создаёт запись через `service.create`, возвращает JSON с ID, URL и датами; успешный статус — 201. |
| `GET /v1/snippets/{snippet_id}` | Передаёт ID в `service.get`, возвращает JSON с текстом и метаданными; успешный статус — 200. |
| `GET /s/{snippet_id}` | Берёт из результата `get()` только `content`, возвращает его как `text/plain`. Это публичный URL из ответа создания. |

В `create_rest` параметр `body` берётся из JSON, а `idempotency_key` — из HTTP-заголовка `Idempotency-Key`: `Header` по умолчанию преобразует подчёркивания имени в дефисы. Аннотация `str | None` допускает строку или отсутствие значения. В URL фигурные скобки означают переменную часть пути. Для успешного идемпотентного повтора POST тоже возвращается 201, поскольку статус задан на маршруте.

**4. Запуск Uvicorn.** `rest_server` и `rest_thread` сначала равны `None`. `global` позволяет функции `start_rest()` заменить эти переменные на уровне ноутбука. Проверка `rest_server.started` пропускает повторный запуск, если известный функции сервер уже работает.

`uvicorn.Config` задаёт приложение, локальный адрес `127.0.0.1`, порт и настройки логов. Uvicorn обслуживает FastAPI. Сервер запускается через `threading.Thread(..., daemon=True)`, чтобы длительная работа сервера не удерживала выполнение ячейки; фоновый поток не мешает завершению процесса Python.

После запуска цикл до 100 раз проверяет `started`, делая паузы по 0,05 секунды — это примерно 5 секунд ожидания. Если готовность не наступила, возникает `RuntimeError`. Затем печатаются адрес API и адрес `/docs`, где FastAPI предоставляет интерактивную документацию. Флаг `daemon` не заменяет корректную остановку из последней ячейки.


## 5. gRPC server
Protocol Buffers messages, the same storage and validation, and status mappings for errors.

In [ ]:
import grpc
from concurrent import futures

GRPC_ERROR_CODES = {
    404: grpc.StatusCode.NOT_FOUND,
    409: grpc.StatusCode.ALREADY_EXISTS,
    413: grpc.StatusCode.RESOURCE_EXHAUSTED,
    422: grpc.StatusCode.INVALID_ARGUMENT,
    503: grpc.StatusCode.UNAVAILABLE,
}

class PastebinGrpc(pb_grpc.PastebinServicer):
    def CreateSnippet(self, request, context):
        try:
            result = service.create(request.content, request.title, request.language or 'plaintext',
                                    request.ttl_seconds or DEFAULT_TTL, request.idempotency_key or None)
            return pb.CreateSnippetResponse(**result)
        except ServiceError as e:
            context.abort(GRPC_ERROR_CODES.get(e.status, grpc.StatusCode.INTERNAL), e.message)

    def GetSnippet(self, request, context):
        try:
            return pb.GetSnippetResponse(**service.get(request.id))
        except ServiceError as e:
            context.abort(GRPC_ERROR_CODES.get(e.status, grpc.StatusCode.INTERNAL), e.message)

grpc_server = grpc.server(futures.ThreadPoolExecutor(max_workers=10),
                          options=[('grpc.max_receive_message_length', 2 * MAX_BYTES),
                                   ('grpc.max_send_message_length', 2 * MAX_BYTES)])
pb_grpc.add_PastebinServicer_to_server(PastebinGrpc(), grpc_server)
if grpc_server.add_insecure_port(f'127.0.0.1:{GRPC_PORT}') == 0:
    raise RuntimeError('Could not bind gRPC port')
grpc_server.start()
print(f'gRPC listening at 127.0.0.1:{GRPC_PORT}')

### Разбор блока 5: сервер gRPC

**Задача блока:** открыть второй сетевой интерфейс к тому же объекту `service`. Здесь передаются сообщения Protocol Buffers и вызываются именованные методы, объявленные в `PROTO`.

**1. Соответствие ошибок.** У gRPC своя система статусов, поэтому `GRPC_ERROR_CODES` переводит числовые статусы сервиса:

| Статус сервиса | Статус gRPC | Причина в этом приложении |
|---|---|---|
| 404 | `NOT_FOUND` | Запись отсутствует или истекла. |
| 409 | `ALREADY_EXISTS` | Ключ повторно использован с другими параметрами. |
| 413 | `RESOURCE_EXHAUSTED` | Текст превышает допустимый размер. |
| 422 | `INVALID_ARGUMENT` | Данные не проходят проверку. |
| 503 | `UNAVAILABLE` | Не удалось выделить уникальный ID. |

**2. Реализация RPC.** `PastebinGrpc` наследуется от сгенерированного `PastebinServicer`. Имена `CreateSnippet` и `GetSnippet` соответствуют контракту. `request` содержит уже разобранное сообщение, `context` позволяет управлять текущим RPC, в том числе завершать его с ошибкой.

`CreateSnippet` извлекает поля и вызывает `service.create`. Выражение `x or значение` подставляет правую часть, если `x` ложно: пустой язык заменяется на `plaintext`, нулевой TTL — на 7 дней, пустой ключ — на `None`. Это связано со скалярными значениями по умолчанию в `proto3`. Важный нюанс: явно переданный TTL `0` здесь тоже станет значением по умолчанию, тогда как REST передаст ноль сервису и получит 422. Пустой язык также обрабатывается транспортами по-разному.

`pb.CreateSnippetResponse(**result)` разворачивает словарь в именованные аргументы конструктора сообщения. Например, ключ `id` становится аргументом `id=...`. `GetSnippet` аналогично передаёт ID в `service.get` и строит `GetSnippetResponse`.

При `ServiceError` вызов `context.abort(...)` прекращает RPC с подходящим статусом и текстом ошибки. `.get(e.status, grpc.StatusCode.INTERNAL)` задаёт запасной статус, если в таблице нет нужного кода.

**3. Создание и запуск сервера.** `ThreadPoolExecutor(max_workers=10)` задаёт пул до десяти рабочих потоков для обработчиков. Параметры `grpc.max_receive_message_length` и `grpc.max_send_message_length` разрешают сообщения до 2 МиБ, учитывая метаданные и служебные поля. Бизнес-лимит самого `content` остаётся 1 МиБ — его отдельно проверяет сервис.

`add_PastebinServicer_to_server(...)` регистрирует обработчик. `add_insecure_port(...)` привязывает сервер к локальному адресу без TLS; нулевой результат означает, что порт привязать не удалось. `grpc_server.start()` запускает обработку в фоне и возвращает управление ячейке.

Оба транспорта используют одну базу и общий сервис, поэтому созданную через gRPC запись можно читать через REST. Поле `url` у ответа gRPC содержит обычную HTTP-ссылку на публичный REST-маршрут.


## 6. Client examples and smoke tests
Both clients send equivalent requests. The gRPC service returns a browser-friendly REST URL.

In [ ]:
import httpx

channel = grpc.insecure_channel(f'127.0.0.1:{GRPC_PORT}')
grpc.channel_ready_future(channel).result(timeout=5)
stub = pb_grpc.PastebinStub(channel)

key = str(uuid.uuid4())
request = pb.CreateSnippetRequest(content='print("Hello from SIS2")', title='demo',
                                  language='python', ttl_seconds=3600, idempotency_key=key)
created = stub.CreateSnippet(request, timeout=2)
created_again = stub.CreateSnippet(request, timeout=2)
assert created.id == created_again.id, 'idempotency failed'
read_grpc = stub.GetSnippet(pb.GetSnippetRequest(id=created.id), timeout=2)
assert read_grpc.content == request.content

with httpx.Client(timeout=2) as client:
    rest_response = client.post(f'{BASE_URL}/v1/snippets', json={
        'content': 'print("Hello from REST")', 'title': 'demo',
        'language': 'python', 'ttl_seconds': 3600},
        headers={'Idempotency-Key': str(uuid.uuid4())})
    assert rest_response.status_code == 201, rest_response.text
    rest_created = rest_response.json()
    rest_read = client.get(f'{BASE_URL}/v1/snippets/{rest_created["id"]}')
    assert rest_read.status_code == 200
    assert rest_read.json()['content'] == 'print("Hello from REST")'
    public_read = client.get(f'{BASE_URL}/s/{rest_created["id"]}')
    assert public_read.text == 'print("Hello from REST")'

print('PASS: gRPC create, gRPC read, idempotent retry, REST create/read, public URL')
print('gRPC ID:', created.id, '| REST ID:', rest_created['id'])
print('Public URL:', rest_created['url'])

### Разбор блока 6: клиенты и базовые проверки

**Задача блока:** обратиться к работающим серверам по сети и проверить основной сценарий «создать → прочитать», повтор запроса и публичную ссылку. Для выполнения нужны успешно запущенные предыдущие ячейки.

**1. Подключение gRPC.** `grpc.insecure_channel(...)` создаёт канал без TLS. `channel_ready_future(channel).result(timeout=5)` ждёт готовности канала до 5 секунд. `PastebinStub(channel)` — сгенерированный клиентский объект, через который вызываются RPC, например `stub.CreateSnippet(...)`.

**2. Запрос создания.** `uuid.uuid4()` создаёт случайный UUID для ключа идемпотентности. В `CreateSnippetRequest` передаются пример текста, заголовок, язык Python и TTL 3600 секунд, то есть один час. Текст `print(...)` хранится как строка: сервис его не исполняет.

Один запрос отправляется дважды. `assert created.id == created_again.id` проверяет одинаковый ID, то есть поведение идемпотентного повтора. Затем `GetSnippetRequest` передаёт ID, а следующая проверка сравнивает прочитанный текст с исходным. `timeout=2` ограничивает время ожидания каждого RPC; автоматического повторения после ошибки в этом коде нет — второй вызов написан явно.

**3. Запросы REST через HTTPX.** Контекст `with httpx.Client(timeout=2) as client` открывает переиспользуемый HTTP-клиент и закрывает его при выходе. Параметр `json={...}` сериализует словарь в JSON-тело, `headers={...}` передаёт ключ в HTTP-заголовке. HTTPX применяет таймауты к сетевым операциям; это не обязательно общий предел в 2 секунды на весь сценарий.

POST создаёт другую запись и должен вернуть 201. `rest_response.json()` разбирает тело ответа в Python-структуру, откуда берётся ID. Далее GET проверяет статус 200 и содержимое JSON, а запрос `/s/{id}` — текст публичной страницы. Здесь gRPC- и REST-примеры используют разные строки и разные ключи, поэтому у них отдельные записи.

**4. Как читать результат.** `assert условие, сообщение` выбрасывает `AssertionError`, если условие ложно; в обычном режиме выполнение ячейки на этом месте останавливается. У первой REST-проверки сообщением служит `rest_response.text`, чтобы видеть причину неожиданного статуса. Если выполнение дошло до `PASS`, все предшествующие утверждения прошли; затем печатаются ID и URL.

Это базовые проверки работоспособности, а не измерение скорости или устойчивости под нагрузкой. Наличие таймаутов не означает, что проверен сценарий их срабатывания; отдельные проверки истечения TTL и сетевых сбоев здесь отсутствуют.


## 7. Additional functional checks
Validation, duplicate-key conflict, and missing IDs. **Step 4** will add dedicated timeout/retry tests; **Step 5** will add latency and payload-size benchmarks.

In [ ]:
from fastapi.testclient import TestClient

with TestClient(app) as client:
    assert client.get('/v1/snippets/NOTFOUND').status_code == 404
    assert client.post('/v1/snippets', json={'content': '', 'ttl_seconds': 60}).status_code == 422
    assert client.post('/v1/snippets', json={'content': 'a' * (MAX_BYTES + 1), 'ttl_seconds': 60}).status_code == 413
    conflict_key = str(uuid.uuid4())
    h = {'Idempotency-Key': conflict_key}
    a = client.post('/v1/snippets', json={'content': 'first', 'ttl_seconds': 60}, headers=h)
    b = client.post('/v1/snippets', json={'content': 'different', 'ttl_seconds': 60}, headers=h)
    assert a.status_code == 201 and b.status_code == 409
print('PASS: 404, empty content, oversize content, conflicting idempotency key')

### Разбор блока 7: проверки ошибочных запросов

**Задача блока:** убедиться, что REST API возвращает ожидаемые ошибки и защищает ключ идемпотентности от использования с другим запросом.

`TestClient(app)` обращается непосредственно к FastAPI-приложению внутри процесса, без сетевого запроса на порт Uvicorn. При этом обработчики и объект `service` настоящие, поэтому проверки работают с той же SQLite-базой. `with` управляет жизненным циклом тестового клиента.

| Проверка | Ожидаемый результат и причина |
|---|---|
| GET с ID `NOTFOUND` | 404: запись с таким ID не найдена. |
| POST с пустым `content` | 422: общий сервис запрещает пустой текст. TTL 60 допустим. |
| POST с `'a' * (MAX_BYTES + 1)` | 413: каждый ASCII-символ `a` занимает один байт, поэтому текст превышает лимит ровно на байт. |
| Два POST с одним ключом, но текстами `first` и `different` | Первый возвращает 201, второй — 409: ключ уже связан с другим отпечатком параметров. |

В проверке конфликта `conflict_key` — новый UUID, а `h` — словарь заголовков, который передаётся в оба запроса. Переменные `a` и `b` содержат HTTP-ответы. Выражение `a.status_code == 201 and b.status_code == 409` требует, чтобы оба условия выполнялись.

Итоговая строка `PASS` печатается только после успешного выполнения всех `assert`. Эти проверки дополняют успешные сценарии предыдущего блока, но напрямую проверяют ошибки только REST. В этом блоке нет отдельных проверок ошибок gRPC, границ TTL, истечения срока или производительности; таймауты/повторы при сбое и бенчмарки указаны в ноутбуке как дальнейшая работа.


## 8. Shut down servers
Run when finished. To start again without re-running everything, restart the kernel and run cells top-to-bottom.

In [ ]:
channel.close()
grpc_server.stop(grace=1).wait(timeout=3)
if rest_server is not None:
    rest_server.should_exit = True
    rest_thread.join(timeout=5)
print('Servers stopped')

### Разбор блока 8: завершение работы

**Задача блока:** закрыть клиентский канал и остановить два фоновых сервера, чтобы завершить работу и освободить порты.

1. `channel.close()` закрывает клиентский канал gRPC, созданный в блоке 6. Это не останавливает сам сервер.
2. `grpc_server.stop(grace=1)` запрещает новые RPC и даёт текущим до одной секунды на завершение, после чего незавершённые вызовы прерываются. Возвращённый объект позволяет ждать остановки; `.wait(timeout=3)` ждёт не больше трёх секунд.
3. Если REST-сервер существует, `rest_server.should_exit = True` подаёт Uvicorn сигнал завершиться. `rest_thread.join(timeout=5)` ждёт окончания его потока до пяти секунд.
4. `print('Servers stopped')` выводит сообщение после этих действий. Результат ожидания gRPC и состояние REST-потока после таймаута здесь не проверяются, поэтому одна эта строка не является строгой проверкой успешной остановки.

Ячейка предполагает, что предыдущие блоки уже создали `channel`, `grpc_server` и REST-переменные. Если запуск прервался раньше, обращение к ещё не созданной переменной может вызвать `NameError`.

Остановка не удаляет SQLite-базу, `.proto` и сгенерированные модули: они остаются в рабочей папке. Для полного повторного запуска следуй инструкции ноутбука: перезапусти ядро и выполни ячейки сверху вниз.
